# 4.3 — Decision Trees

**Type:** Classification (and Regression)  
**Core idea:** Split data using a series of yes/no questions learned from the data. Each split maximises class purity.  
**When to use:** When you need interpretability, mixed feature types, no scaling required, quick baseline.

---

## Real World Problem: Employee Attrition Prediction
### HR Department — IT Company, Bangalore

An IT company in Bangalore is losing employees faster than they can hire.  
The HR team wants to **predict which employees are likely to leave** — so they can intervene early (raise, promotion, better project assignment).

**Stakes:**
- Replacing one engineer costs ₹3–5 lakhs (recruitment + training + lost productivity)
- If we can identify at-risk employees early → retain them → save money
- **False Negative** (miss someone who leaves) → company loses them
- **False Positive** (flag someone who stays) → unnecessary intervention cost

**Why Decision Tree fits here:**  
HR managers need to **explain** to leadership *why* an employee is flagged as at-risk.  
"The model says so" is not acceptable. "Employees with salary < ₹8L, overtime = Yes, and no promotion in 3 years have 78% attrition rate" — that is.

## How Decision Trees Work — The Intuition

**Analogy:** A senior HR manager in Kerala has seen hundreds of resignations. She's developed a mental flowchart:
```
Does the employee do overtime regularly?
├── NO  → Is their salary above ₹10L?
│         ├── YES → likely STAYS
│         └── NO  → Is job satisfaction low?
│                   ├── YES → likely LEAVES
│                   └── NO  → likely STAYS
└── YES → Is their last promotion > 2 years ago?
          ├── YES → very likely LEAVES
          └── NO  → likely STAYS
```

The Decision Tree learns this flowchart automatically from data.

### How it finds the best split — Gini Impurity

At every node, the tree asks: *which feature and threshold best separates the classes?*

It measures this using **Gini Impurity**:
```
Gini = 1 - (p_stays² + p_leaves²)
```
- Gini = 0.0 → node is perfectly pure (all one class) ← BEST
- Gini = 0.5 → node is perfectly mixed (50/50) ← WORST

The tree picks the split that reduces Gini the most → moves from mixed nodes toward pure nodes.

### Why overfitting is the main danger
A fully grown tree memorises every training sample → 100% train accuracy, terrible test accuracy.  
Solution: control depth with `max_depth`, `min_samples_leaf`, etc.

## Step 0 — Import Libraries

In [ ]:
# numpy — numerical operations, random data generation
import numpy as np

# pandas — load, clean, manipulate tabular data
import pandas as pd

# matplotlib + seaborn — plotting
import matplotlib.pyplot as plt
import seaborn as sns

# DecisionTreeClassifier — the decision tree model
from sklearn.tree import DecisionTreeClassifier

# plot_tree — visualise the tree structure (what splits were learned)
# export_text — print the tree as a text-based flowchart
from sklearn.tree import plot_tree, export_text

# train_test_split — split into train and test
# cross_val_score — k-fold cross validation
# StratifiedKFold — preserves class ratio in each fold
# GridSearchCV — try all combinations of hyperparameters to find the best
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, GridSearchCV
)

# LabelEncoder — convert text categories to numbers (for target column)
from sklearn.preprocessing import LabelEncoder

# Evaluation metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, recall_score, precision_score
)

# Note: Decision Trees do NOT need StandardScaler
# They make decisions based on thresholds (>, <) not distances
# So feature scale doesn't affect the result

np.random.seed(42)
print("All libraries loaded.")

## Step 1 — Gini Impurity From Scratch (Understand Before Using sklearn)

In [ ]:
# --- Gini Impurity from scratch ---
# Measures how mixed a node is
# Gini = 0   → pure node (all one class) — perfect
# Gini = 0.5 → maximally mixed (50/50)   — worst

def gini_impurity(y):
    """
    Calculate Gini impurity for an array of labels.
    Formula: 1 - sum(p_i^2) for each class i
    """
    if len(y) == 0:
        return 0
    
    # Count unique classes and their proportions
    _, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)  # p_i for each class
    
    # Gini = 1 - sum of squared proportions
    return 1 - np.sum(proportions ** 2)


def weighted_gini(y_left, y_right):
    """
    Calculate weighted Gini after a split.
    The tree picks the split that minimises this value.
    """
    n = len(y_left) + len(y_right)            # total samples
    w_left  = len(y_left)  / n                # weight of left child
    w_right = len(y_right) / n                # weight of right child
    
    # Weighted average of children's Gini
    return w_left * gini_impurity(y_left) + w_right * gini_impurity(y_right)


# Demonstrate with examples
pure_node     = np.array([0, 0, 0, 0, 0])    # all one class
mixed_node    = np.array([0, 0, 1, 1, 0])    # mixed
half_half     = np.array([0, 0, 0, 1, 1, 1]) # 50/50

print("Gini Impurity Examples:")
print(f"  Pure node  {pure_node}  → Gini = {gini_impurity(pure_node):.4f}")
print(f"  Mixed node {mixed_node} → Gini = {gini_impurity(mixed_node):.4f}")
print(f"  50/50 node {half_half}  → Gini = {gini_impurity(half_half):.4f}")

print("\nSplit comparison:")
# Good split: separates the classes well
left_good  = np.array([0, 0, 0, 0])   # mostly stays
right_good = np.array([1, 1, 1, 1])   # mostly leaves
print(f"  Good split  → Weighted Gini = {weighted_gini(left_good, right_good):.4f}")

# Bad split: both sides still mixed
left_bad   = np.array([0, 1, 0, 1])
right_bad  = np.array([0, 1, 0, 1])
print(f"  Bad split   → Weighted Gini = {weighted_gini(left_bad, right_bad):.4f}")
print("\nLower weighted Gini = better split → tree picks the good one")

In [ ]:
# Visualise Gini impurity vs class proportion
# This shows WHY the tree tries to reach Gini=0 at leaves

p = np.linspace(0, 1, 100)   # proportion of class 1 (from 0% to 100%)
gini = 2 * p * (1 - p)       # Gini for binary classification: 1 - (p² + (1-p)²) = 2p(1-p)

plt.figure(figsize=(7, 4))
plt.plot(p, gini, color='steelblue', linewidth=2)
plt.axvline(0.5, color='red', linestyle='--', label='50/50 split — worst (Gini=0.5)')
plt.axhline(0,   color='green', linestyle='--', label='Pure node — best (Gini=0.0)')
plt.fill_between(p, gini, alpha=0.1, color='steelblue')
plt.title("Gini Impurity vs Class Proportion")
plt.xlabel("Proportion of class 1 in node")
plt.ylabel("Gini Impurity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2 — Create the Employee Attrition Dataset

In [ ]:
np.random.seed(42)
n = 800   # 800 employees

# --- Generate features ---

# Monthly salary in lakhs per annum (₹ LPA)
# Most IT employees earn 5-20 LPA, skewed right (few earn very high)
salary = np.random.normal(12, 5, n).clip(4, 40)

# Years at the company (tenure)
# New joinees and very long-tenured employees both have different attrition patterns
tenure = np.random.randint(1, 15, n)

# Job satisfaction score (1=very low, 4=very high) — self-reported in HR survey
satisfaction = np.random.randint(1, 5, n)

# Overtime: 1 = works overtime regularly, 0 = does not
overtime = np.random.choice([0, 1], n, p=[0.6, 0.4])  # 40% work overtime

# Years since last promotion
years_no_promo = np.random.randint(0, 8, n)

# Number of companies worked at before joining
# High number = more comfortable switching jobs = higher attrition risk
num_prev_companies = np.random.randint(0, 7, n)

# Distance from home to office (km) — longer commute = less happy
distance_km = np.random.randint(1, 50, n)

# Work-life balance rating (1=very bad, 4=very good)
work_life = np.random.randint(1, 5, n)

# --- Generate target: attrition (1=left, 0=stayed) ---
# Based on realistic HR patterns:
# High risk: overtime + low satisfaction + low salary + no promotion + long commute
risk_score = (
    (overtime == 1).astype(int) * 2         +  # overtime is biggest predictor
    (satisfaction <= 2).astype(int) * 2     +  # low satisfaction
    (salary < 8).astype(int) * 2            +  # underpaid
    (years_no_promo >= 4).astype(int) * 1   +  # no promotion for long
    (distance_km > 30).astype(int) * 1      +  # long commute
    (work_life <= 2).astype(int) * 1        +  # poor work-life balance
    (num_prev_companies >= 4).astype(int) * 1  # job hopper
)

# Convert risk score to probability, then to binary label
prob_leave = risk_score / risk_score.max()          # normalise to 0-1
attrition  = (np.random.rand(n) < prob_leave).astype(int)  # probabilistic label

# --- Build DataFrame ---
df = pd.DataFrame({
    'salary_lpa':          salary.round(1),
    'tenure_years':        tenure,
    'satisfaction':        satisfaction,
    'overtime':            overtime,
    'years_no_promotion':  years_no_promo,
    'prev_companies':      num_prev_companies,
    'distance_km':         distance_km,
    'work_life_balance':   work_life,
    'attrition':           attrition        # target
})

print("Dataset shape:", df.shape)
print(f"\nAttrition rate: {df['attrition'].mean()*100:.1f}%")
print(f"Employees who left:   {df['attrition'].sum()}")
print(f"Employees who stayed: {(df['attrition']==0).sum()}")
print("\nFirst 5 rows:")
df.head()

## Step 3 — EDA (Explore Before Modelling)

In [ ]:
# Always inspect first — never jump straight to modelling

print("=== Basic Info ===")
print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nStatistics:")
df.describe().round(2)

In [ ]:
# Attrition rate by key categorical features
# This gives HR immediate insights even before the model

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Attrition by overtime
overtime_rate = df.groupby('overtime')['attrition'].mean() * 100
axes[0].bar(['No Overtime', 'Overtime'],
            overtime_rate.values,
            color=['steelblue', 'coral'])
axes[0].set_title('Attrition Rate by Overtime')
axes[0].set_ylabel('Attrition %')
for i, v in enumerate(overtime_rate.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Attrition by satisfaction
sat_rate = df.groupby('satisfaction')['attrition'].mean() * 100
axes[1].bar([f'Score {i}' for i in sat_rate.index],
            sat_rate.values, color='mediumseagreen')
axes[1].set_title('Attrition Rate by Satisfaction')
axes[1].set_ylabel('Attrition %')

# Attrition by work-life balance
wl_rate = df.groupby('work_life_balance')['attrition'].mean() * 100
axes[2].bar([f'WLB {i}' for i in wl_rate.index],
            wl_rate.values, color='mediumpurple')
axes[2].set_title('Attrition Rate by Work-Life Balance')
axes[2].set_ylabel('Attrition %')

plt.suptitle('Attrition Rates by Key Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions — leavers vs stayers

numeric_features = ['salary_lpa', 'tenure_years', 'years_no_promotion',
                    'distance_km', 'prev_companies']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, feat in zip(axes, numeric_features):
    # Stayers in blue
    df[df['attrition'] == 0][feat].hist(
        ax=ax, alpha=0.6, color='steelblue', label='Stayed', bins=15
    )
    # Leavers in coral
    df[df['attrition'] == 1][feat].hist(
        ax=ax, alpha=0.6, color='coral', label='Left', bins=15
    )
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions — Left vs Stayed', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with attrition — quick feature ranking
corr = df.corr()['attrition'].drop('attrition').sort_values(ascending=False)

print("Feature correlation with attrition:")
print(corr.round(3))

plt.figure(figsize=(8, 4))
colors = ['coral' if c > 0 else 'steelblue' for c in corr]
corr.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Attrition\n(positive = higher value → more likely to leave)')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

## Step 4 — Prepare Data (No Scaling Needed!)

In [ ]:
# Split into features (X) and target (y)

# X = all columns except the target
X = df.drop(columns=['attrition'])

# y = target column
y = df['attrition']

# IMPORTANT NOTE:
# We do NOT apply StandardScaler here
# Decision Trees split on thresholds (salary < 8 LPA?)
# Scaling doesn't change the ordering or relative values
# So it has NO EFFECT on the tree's decisions
# This is a key advantage over KNN and Logistic Regression

print("X shape:", X.shape)   # (800, 8)
print("y shape:", y.shape)   # (800,)
print("\nFeatures:", list(X.columns))

# Train/test split — stratified to preserve attrition rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% = 160 employees for testing
    random_state=42,
    stratify=y            # same attrition % in both sets
)

print(f"\nTrain: {X_train.shape[0]} employees")
print(f"Test:  {X_test.shape[0]} employees")
print(f"\nAttrition rate — Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

## Step 5 — Train a Decision Tree (and See the Overfitting Problem)

In [ ]:
# First: fully grown tree — no depth limit
# This will show classic overfitting

dt_overfit = DecisionTreeClassifier(
    random_state=42
    # No max_depth — tree grows until every leaf is pure
    # This means it will memorise every training sample
)

dt_overfit.fit(X_train, y_train)

# Score on train and test
train_acc = dt_overfit.score(X_train, y_train)   # how well it memorised training data
test_acc  = dt_overfit.score(X_test, y_test)     # how well it generalises to new data

print("=== Fully Grown Tree (No Depth Limit) ===")
print(f"Tree depth:        {dt_overfit.get_depth()}")
print(f"Number of leaves:  {dt_overfit.get_n_leaves()}")
print(f"Train accuracy:    {train_acc*100:.1f}%")
print(f"Test accuracy:     {test_acc*100:.1f}%")
print()
print("Notice: near 100% train but much lower test — classic OVERFITTING")
print("The tree memorised every employee instead of learning general patterns")

## Step 6 — Find the Best max_depth (Prevent Overfitting)

In [ ]:
# Try different max_depth values — find the sweet spot
# Too shallow = underfitting (misses patterns)
# Too deep = overfitting (memorises noise)
# Just right = generalises well

depths = range(1, 20)       # try depth 1 through 19
train_scores = []           # training accuracy per depth
test_scores  = []           # test accuracy per depth
cv_f1_scores = []           # cross-validated F1 per depth (most reliable)

# 5-fold stratified CV for reliable F1 estimation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for d in depths:
    dt = DecisionTreeClassifier(
        max_depth=d,
        random_state=42
    )
    dt.fit(X_train, y_train)

    # Training accuracy (will always increase with depth)
    train_scores.append(dt.score(X_train, y_train))

    # Test accuracy (goes up then down — peaks at best depth)
    test_scores.append(dt.score(X_test, y_test))

    # Cross-validated F1 — most reliable, use this to pick best depth
    # We use F1 because attrition is somewhat imbalanced (not 50/50)
    cv_f1 = cross_val_score(dt, X_train, y_train, cv=cv, scoring='f1').mean()
    cv_f1_scores.append(cv_f1)

# Best depth = highest CV F1
best_depth = list(depths)[np.argmax(cv_f1_scores)]
print(f"Best depth by cross-validated F1: {best_depth}")
print(f"CV F1 at best depth: {max(cv_f1_scores):.4f}")

# Plot all three curves
plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, label='Train Accuracy', color='steelblue', marker='o', markersize=4)
plt.plot(depths, test_scores,  label='Test Accuracy',  color='coral',     marker='o', markersize=4)
plt.plot(depths, cv_f1_scores, label='CV F1 (reliable)', color='green',   marker='s', markersize=4, linestyle='--')
plt.axvline(best_depth, color='black', linestyle=':', label=f'Best depth = {best_depth}')

plt.title('Choosing max_depth — Overfitting vs Underfitting')
plt.xlabel('Tree Depth (max_depth)')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nWhat the plot shows:")
print("  Train accuracy always rises with depth (memorises more)")
print("  Test accuracy rises then falls (overfits after best depth)")
print("  CV F1 is the most reliable signal — use it to pick depth")

## Step 7 — GridSearchCV (Find the Best Combination of Parameters)

In [ ]:
# max_depth alone isn't the only parameter that matters
# We also have min_samples_leaf and criterion
#
# GridSearchCV tries ALL combinations and returns the best one
# This is called hyperparameter tuning
#
# Parameter grid — all combinations will be tried
param_grid = {
    'max_depth':         [3, 4, 5, 6, 7, 8],    # how deep the tree can grow
    'min_samples_leaf':  [5, 10, 15, 20],        # min samples needed in each leaf
    # Higher min_samples_leaf → simpler tree → less overfitting
    'criterion':         ['gini', 'entropy']     # how to measure split quality
    # gini = Gini impurity (faster)
    # entropy = Information Gain (sometimes slightly better)
}

# Total combinations: 6 × 4 × 2 = 48 combinations
# Each is evaluated with 5-fold CV → 48 × 5 = 240 models trained

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',         # optimise for F1 score (recall + precision balance)
    n_jobs=-1,            # use all CPU cores (faster)
    verbose=0
)

grid_search.fit(X_train, y_train)

print("Best parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV F1 score: {grid_search.best_score_:.4f}")

## Step 8 — Train Final Model with Best Parameters

In [ ]:
# Use the best parameters found by GridSearchCV
best_dt = grid_search.best_estimator_   # this is already fitted on full training data

# Predict labels (0=stays, 1=leaves)
y_pred = best_dt.predict(X_test)

# Predict probabilities — column 1 = probability of leaving
y_prob = best_dt.predict_proba(X_test)[:, 1]

print("=== Final Decision Tree — Classification Report ===")
print(classification_report(
    y_test, y_pred,
    target_names=['Stayed (0)', 'Left (1)']
))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Train accuracy: {best_dt.score(X_train, y_train)*100:.1f}%")
print(f"Test accuracy:  {best_dt.score(X_test, y_test)*100:.1f}%")
print(f"\nGap (overfit check): {(best_dt.score(X_train, y_train) - best_dt.score(X_test, y_test))*100:.1f}%")
print("(Gap < 5% is fine. Gap > 10% = still overfitting, increase min_samples_leaf)")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=['Predicted Stay', 'Predicted Leave'],
    yticklabels=['Actual Stay', 'Actual Leave']
)
plt.title('Confusion Matrix — Decision Tree')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (stayed, correctly kept):         {tn}")
print(f"False Positives (stayed, wrongly flagged):        {fp}  ← unnecessary HR intervention")
print(f"False Negatives (left, we missed):                {fn}  ← employee lost, cost ₹3-5L")
print(f"True Positives  (left, correctly flagged):        {tp}  ← HR can intervene early")
print(f"\nOf {fn+tp} employees who left, we caught {tp} ({tp/(fn+tp)*100:.1f}%) in time")

## Step 9 — Visualise the Tree (The Unique Advantage of Decision Trees)

In [ ]:
# This is what makes Decision Trees special — you can SEE the logic
# No other algorithm gives you this level of transparency

# Get best depth to limit visual complexity
viz_depth = min(best_dt.get_depth(), 4)  # show max 4 levels for readability

# Create a shallow version just for visualisation
dt_viz = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
    criterion=grid_search.best_params_['criterion'],
    random_state=42
)
dt_viz.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(
    dt_viz,
    feature_names=X.columns.tolist(),    # column names
    class_names=['Stayed', 'Left'],       # target class names
    filled=True,                          # colour nodes by class
    rounded=True,                         # rounded boxes
    fontsize=9,
    impurity=True,                        # show Gini at each node
    proportion=False                      # show sample counts not proportions
)
plt.title('Decision Tree — Employee Attrition\n(Orange = likely leaves, Blue = likely stays)', 
          fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print the tree as a text flowchart — easier to read and copy into a report
# This is what you'd share with HR leadership

print("=== Decision Tree Rules (Text Format) ===")
print("Read as: if condition is True → go left, else → go right")
print()
rules = export_text(
    dt_viz,
    feature_names=list(X.columns),
    max_depth=4         # show top 4 levels
)
print(rules)
print("class: 0 = employee stays, class: 1 = employee leaves")

## Step 10 — Feature Importance (What Does HR Need to Focus On?)

In [ ]:
# Feature importance = how much each feature reduced Gini impurity across all splits
# Higher = more useful for predicting attrition
# This is UNIQUE to tree-based models — KNN and Logistic Regression can't do this as cleanly

importances = best_dt.feature_importances_    # array of importance scores
feature_names = X.columns.tolist()

# Create a sorted DataFrame for easy reading
importance_df = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances (how much each feature contributes to predictions):")
print(importance_df.round(4).to_string(index=False))

# Visualise
plt.figure(figsize=(9, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(importance_df)))
plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
plt.xlabel('Importance Score (Gini reduction)')
plt.title('Feature Importances — What Drives Attrition?')
plt.gca().invert_yaxis()   # most important at top
plt.tight_layout()
plt.show()

top_feature = importance_df.iloc[0]['Feature']
print(f"\nMost important predictor of attrition: {top_feature}")
print("HR should address this first for the biggest retention impact.")

## Step 11 — Threshold Tuning for HR Context

In [ ]:
# Should HR use threshold = 0.5 or adjust it?
#
# Cost analysis:
# - Missing an employee who leaves (FN): costs ₹3–5 lakhs to replace them
# - Flagging someone who stays (FP): costs ~₹10,000 for an intervention (1-on-1 with manager)
# - Missing a leaver is 30–50x more expensive than a false alarm
#
# Therefore: LOWER the threshold to catch more leavers
# Even if we get more false alarms, the cost is acceptable

thresholds   = np.arange(0.10, 0.90, 0.05)
recalls      = []
precisions   = []
f1s          = []
missed_count = []    # employees who left but we didn't flag
flagged_fp   = []    # employees who stayed but were wrongly flagged

for thresh in thresholds:
    # Convert probability to class label using this threshold
    y_thresh = (y_prob >= thresh).astype(int)

    recalls.append(recall_score(y_test, y_thresh, zero_division=0))
    precisions.append(precision_score(y_test, y_thresh, zero_division=0))
    f1s.append(f1_score(y_test, y_thresh, zero_division=0))

    cm_t = confusion_matrix(y_test, y_thresh)
    missed_count.append(cm_t[1][0])   # FN: leavers we missed
    flagged_fp.append(cm_t[0][1])     # FP: stayers we wrongly flagged

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: precision/recall/F1 curves
axes[0].plot(thresholds, recalls,    color='coral',     label='Recall',    marker='o', markersize=3)
axes[0].plot(thresholds, precisions, color='steelblue', label='Precision', marker='o', markersize=3)
axes[0].plot(thresholds, f1s,        color='green',     label='F1',        marker='s', markersize=3)
axes[0].axvline(0.5,  color='gray',  linestyle='--', alpha=0.7, label='Default (0.5)')
axes[0].axvline(0.35, color='black', linestyle=':',  alpha=0.8, label='Conservative (0.35)')
axes[0].set_title('Metrics vs Threshold')
axes[0].set_xlabel('Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: business impact in people
axes[1].plot(thresholds, missed_count, color='red',    linewidth=2, label='Leavers missed (FN) — ₹3-5L each')
axes[1].plot(thresholds, flagged_fp,   color='orange', linewidth=2, label='Stayers wrongly flagged (FP) — ₹10k each')
axes[1].axvline(0.5,  color='gray',  linestyle='--', alpha=0.7, label='Default (0.5)')
axes[1].axvline(0.35, color='black', linestyle=':',  alpha=0.8, label='Conservative (0.35)')
axes[1].set_title('Business Impact vs Threshold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Number of employees')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare 0.5 vs 0.35
idx_05  = np.argmin(np.abs(thresholds - 0.50))
idx_035 = np.argmin(np.abs(thresholds - 0.35))

print("=== Threshold Comparison ===")
print(f"\nAt threshold = 0.50 (default):")
print(f"  Leavers missed (FN):          {missed_count[idx_05]}  × ₹4L avg = ₹{missed_count[idx_05]*4}L lost")
print(f"  Stayers wrongly flagged (FP): {flagged_fp[idx_05]}  × ₹10k = ₹{flagged_fp[idx_05]*10000:,}")

print(f"\nAt threshold = 0.35 (conservative for HR):")
print(f"  Leavers missed (FN):          {missed_count[idx_035]}  × ₹4L avg = ₹{missed_count[idx_035]*4}L lost")
print(f"  Stayers wrongly flagged (FP): {flagged_fp[idx_035]}  × ₹10k = ₹{flagged_fp[idx_035]*10000:,}")

## Step 12 — Cross-Validation (Is the Model Stable?)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"=== 5-Fold Cross-Validation ===")
for metric_name in ['accuracy', 'f1', 'roc_auc']:
    scores = cross_val_score(
        best_dt,        # our final tuned model
        X_train,        # ONLY training data — test stays untouched
        y_train,
        cv=cv,
        scoring=metric_name
    )
    print(f"  {metric_name:12s}: {scores.mean():.4f} ± {scores.std():.4f}"
          f"  | Folds: {[round(s,3) for s in scores]}")

print("\nInterpretation:")
print("  std < 0.03 → model is stable and consistent")
print("  std > 0.06 → model is sensitive to which employees end up in train vs test")

## Step 13 — Predict for New Employees (Real-World Usage)

In [ ]:
# Three employees walked into HR for their quarterly review
# Should HR be worried about any of them leaving?

new_employees = pd.DataFrame([
    # salary_lpa, tenure, satisfaction, overtime, yrs_no_promo, prev_companies, distance, wlb
    {'salary_lpa': 6.5, 'tenure_years': 2, 'satisfaction': 1,
     'overtime': 1, 'years_no_promotion': 2, 'prev_companies': 5,
     'distance_km': 35, 'work_life_balance': 1},   # Ravi — every red flag

    {'salary_lpa': 18.0, 'tenure_years': 8, 'satisfaction': 4,
     'overtime': 0, 'years_no_promotion': 1, 'prev_companies': 1,
     'distance_km': 5,  'work_life_balance': 4},   # Priya — very stable

    {'salary_lpa': 10.0, 'tenure_years': 4, 'satisfaction': 2,
     'overtime': 1, 'years_no_promotion': 3, 'prev_companies': 3,
     'distance_km': 20, 'work_life_balance': 2},   # Meena — borderline
])

# Decision Tree does NOT need scaling — use directly
probs = best_dt.predict_proba(new_employees)[:, 1]    # P(leaving)

# Using 0.35 threshold (conservative — we'd rather flag than miss)
decisions = ['AT RISK ⚠️ — Schedule 1:1' if p >= 0.35 else 'STABLE ✓' for p in probs]

print("=== HR Risk Dashboard — Quarterly Review ===")
print(f"{'Employee':<10} {'P(Leave)':>10} {'Status':>28}")
print("-" * 52)
for name, prob, decision in zip(['Ravi', 'Priya', 'Meena'], probs, decisions):
    print(f"{name:<10} {prob*100:>9.1f}%  {decision}")

print("\n=== Explainable Reasons (Unique to Decision Trees) ===")
# Walk through the tree's decision path for each employee
# This is what HR would tell the manager
for i, (name, prob) in enumerate(zip(['Ravi', 'Priya', 'Meena'], probs)):
    emp = new_employees.iloc[i]
    print(f"\n{name} (P={prob*100:.0f}%):")
    if emp['overtime'] == 1:
        print(f"  → Works overtime regularly")
    if emp['satisfaction'] <= 2:
        print(f"  → Low job satisfaction (score: {emp['satisfaction']}/4)")
    if emp['salary_lpa'] < 8:
        print(f"  → Below-market salary (₹{emp['salary_lpa']}L vs market ₹10-12L)")
    if emp['years_no_promotion'] >= 3:
        print(f"  → No promotion in {emp['years_no_promotion']} years")
    if emp['work_life_balance'] <= 2:
        print(f"  → Poor work-life balance (score: {emp['work_life_balance']}/4)")
    if prob < 0.35:
        print(f"  → No major risk factors detected")

## Step 14 — The Unique Business Report (Only Possible with Decision Trees)

In [ ]:
# Print the exact rules HR leadership can act on
# This is what separates Decision Trees from all other models
# You can literally give this to a manager with no ML knowledge

print("=====================================================")
print("  ATTRITION PREDICTION RULES — HR LEADERSHIP REPORT ")
print("      Bangalore IT Division — Q1 2025              ")
print("=====================================================")
print()
print("METHODOLOGY: Decision Tree trained on 800 employees.")
print(f"TEST SET ACCURACY:  {best_dt.score(X_test, y_test)*100:.0f}%")
print(f"ATTRITION RECALL:   {recall_score(y_test, y_pred)*100:.0f}%  (how many leavers we catch)")
print(f"ROC-AUC:            {roc_auc_score(y_test, y_prob):.3f}")
print()
print("TOP ATTRITION DRIVERS (in order of impact):")
for _, row in importance_df.iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"  {row['Feature']:<25} {bar}  ({row['Importance']*100:.1f}%)")
print()
print("RECOMMENDED ACTIONS:")
top3 = importance_df.head(3)['Feature'].tolist()
actions = {
    'overtime':           'Enforce overtime limits. Hire support staff.',
    'satisfaction':       'Quarterly satisfaction surveys + exit interviews.',
    'salary_lpa':         'Benchmark salaries against industry. Adjust for <₹8L.',
    'years_no_promotion': 'Fast-track promotions for employees >3 years without one.',
    'work_life_balance':  'Introduce flexible work. Review workload distribution.',
    'distance_km':        'Provide remote work or transport allowance for >30km.',
    'prev_companies':     'During hiring, consider job-hopping history.',
    'tenure_years':       'Focus retention programs on 1-3 year employees.'
}
for feat in top3:
    if feat in actions:
        print(f"  {feat}: {actions[feat]}")
print()
print("THRESHOLD USED: 0.35 (conservative — flag early, intervene proactively)")
print("=====================================================")

## Final Summary

### What we built
A Decision Tree that predicts which employees are likely to leave an IT company in Bangalore — and **explains why** with human-readable rules.

### Key decisions and why

| Decision | What | Why |
|---|---|---|
| No scaling | Skipped StandardScaler | DT uses thresholds, not distances |
| GridSearchCV | Tuned max_depth + min_samples_leaf + criterion | Single parameter isn't enough |
| Threshold = 0.35 | Lower than default 0.5 | Missing a leaver costs 30-50x more than false alarm |
| export_text | Printed rules as text | HR can read, understand and act on them |
| Feature importance | Ranked what drives attrition | Tells HR where to invest retention budget |

### Decision Tree Pros & Cons

| Pros | Cons |
|---|---|
| Fully explainable — you can trace any decision | Overfits easily without pruning |
| No scaling required | Small data changes → very different tree |
| Handles mixed feature types naturally | Not competitive with ensemble methods (Ch 4.4) |
| Feature importance built-in | Single tree = high variance |
| Fast to train and predict | Poor on very complex non-linear data |

### The core insight to remember years later
Decision Trees are the **foundation of Random Forests** (Chapter 4.4).  
All the weaknesses (overfitting, high variance) are fixed by building many trees and averaging them.  
But the explainability advantage — the rules, feature importance — carries forward.

## Practice Task — Student Exam Pass/Fail Prediction
### Kerala Board Exam — Predict which students need extra support

In [ ]:
# Dataset: 600 students. Features: study hours, attendance %, sleep hours,
# tuition (yes/no), parent education level (1-3), practice tests done
np.random.seed(7)
n = 600

study_hours   = np.random.normal(4, 2, n).clip(0, 10)
attendance    = np.random.normal(75, 15, n).clip(30, 100)
sleep_hours   = np.random.normal(7, 1.5, n).clip(3, 10)
has_tuition   = np.random.choice([0, 1], n, p=[0.5, 0.5])
parent_edu    = np.random.randint(1, 4, n)
practice_tests= np.random.randint(0, 10, n)

risk = (
    (study_hours < 2).astype(int) * 3 +
    (attendance < 60).astype(int) * 2 +
    (practice_tests < 3).astype(int) * 2 +
    (sleep_hours < 5).astype(int) * 1 +
    (has_tuition == 0).astype(int) * 1
)
prob_fail = risk / risk.max()
passed = (np.random.rand(n) > prob_fail).astype(int)

students_df = pd.DataFrame({
    'study_hours':    study_hours.round(1),
    'attendance_pct': attendance.round(1),
    'sleep_hours':    sleep_hours.round(1),
    'has_tuition':    has_tuition,
    'parent_edu':     parent_edu,
    'practice_tests': practice_tests,
    'passed':         passed
})

print("Student dataset shape:", students_df.shape)
print(f"Pass rate: {students_df['passed'].mean()*100:.1f}%")
students_df.head()

In [ ]:
# YOUR CODE HERE

# Step 1: Split into X and y
X_s = 
y_s = 

# Step 2: Train/test split (stratified, 80/20)
X_train_s, X_test_s, y_train_s, y_test_s = 

# Step 3: Try max_depth from 1 to 15
# Plot train accuracy, test accuracy, CV F1
# Find the best depth


# Step 4: Use GridSearchCV to tune max_depth, min_samples_leaf, criterion
# scoring='f1'


# Step 5: Print classification report + ROC-AUC


# Step 6: Print feature importances
# Which factor matters most for passing — study hours or attendance?


# Step 7: Visualise or print the tree rules (export_text)
# What rule does the tree learn for failing students?


# Step 8: Predict for these 2 students
new_students = pd.DataFrame([
    {'study_hours': 1.5, 'attendance_pct': 45, 'sleep_hours': 5,
     'has_tuition': 0, 'parent_edu': 1, 'practice_tests': 1},  # at risk
    {'study_hours': 6.0, 'attendance_pct': 90, 'sleep_hours': 8,
     'has_tuition': 1, 'parent_edu': 3, 'practice_tests': 7},  # likely passes
])
# Predict and print probability of passing for each